# Extract the Topics of Every Hearing Summary

| Item | Description |
|---|---|
| Goal | Ask the model for the main subjects of each of the 2,352 hearing summaries from T1: up to five short neutral phrases per hearing, most central first, one call per hearing. These phrases are the raw material of the codebook built in T2.1.2 to T2.1.4. |
| Input | `hearing_summaries.parquet` from T1.2, the 2,352 summaries; the model Qwen/Qwen3.5-4B from the project cache. |
| Output | In the T2.1 Interim folder: `track1/topics.parquet`, one row per hearing with its ranked topics, and `track1/status/extract.jsonl`, one line per model call with a hash of the prompt, the full output, token counts, and time. |
| Variables | • `topics_json`: the parsed topics of the hearing, each with `rank`, `topic`, and `topic_lint`<br>• `errors`: named parse errors, empty when the output was clean<br>• `prompt_version`: `summary-topics-0.2-dev`, the recorded name of the wording used |
| Run | Ran once on 2026-08-22 on one A40, 96 minutes for 2,352 calls; the outputs shown are from that run. Not rerun: the stored topics are the ones the codebook was built from, and a new run would sample new wording. |


## Step 1. Import packages


In [ ]:
import hashlib
import json
import os
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


## Step 2. Define file locations


In [ ]:
PROJECT_FOLDER = Path.cwd().resolve().parents[1]  # <section>/<subfolder>/ sits two levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("T2 - Topic Extraction and Stance Analysis/T2.1 - Track 1 - Topic Extraction based on Hearing Summaries")
NOTEBOOK_NAME = "T2.1.1 - Production - Extract Topics from Every Summary"  # the notebook filename without .ipynb
RAW_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Raw"
INTERIM_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Interim"
PROCESSED_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Processed"
INPUT_FOLDER = SCRATCH_FOLDER / "T1 - Text Summarization/T1.2 - Process Entire Sample Congressional Hearings" / "Processed"
TRACK1_FOLDER = INTERIM_FOLDER / "track1"
OUT_FILE = TRACK1_FOLDER / "topics.parquet"
LOG_FILE = TRACK1_FOLDER / "status" / "extract.jsonl"
FIGURE_FOLDER = PROJECT_FOLDER / "R - Results/figures" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
TABLE_FOLDER = PROJECT_FOLDER / "R - Results/tables" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
MODEL_CACHE = SCRATCH_FOLDER / "cache" / "huggingface"
ENV_FILE = PROJECT_FOLDER / "T2 - Topic Extraction and Stance Analysis" / ".env"
for folder in (INTERIM_FOLDER, PROCESSED_FOLDER):
    folder.mkdir(parents=True, exist_ok=True)
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(MODEL_CACHE)
# A token is needed only for gated models. Qwen3.5-4B and bge-large-en-v1.5 are open; the lines stay so the cell also serves a gated model.
if ENV_FILE.exists():
    for line in ENV_FILE.read_text().splitlines():
        name, _, value = line.strip().partition("=")
        if name == "HF_TOKEN" and value:
            os.environ["HF_TOKEN"] = value.strip().strip('"').strip("'")


## Step 3. Define the prompt

The prompt asks for topics as short neutral noun phrases. A topic is the policy object itself, never a stance toward it, never a bare agency or bill number. The same phrase rules govern the stance prompt of T2.2, so the two phrase pools obey one grammar. A lint flags any directional word that slips into a phrase; the flag marks the phrase for review and does not remove it.


In [ ]:
("You are an expert reader of United States congressional hearing transcripts. "
          "You report only what the text states.")

TRACK1_PROMPT_VERSION = "summary-topics-0.2-dev"  # versioned apart from the stance prompt; the production run kept this recorded name
TRACK1_MAX_TOPICS = 5

TRACK1_USER = """Below is a summary of one United States congressional hearing.

Task: list the main topics the hearing covered, ordered from most to least central.

A topic is a short and specific neutral noun phrase: the policy, program, regulation, technology, practice, or budget item itself; never directional wording; name the specific object, not the agency or field around it. An agency, company, or country is never a topic by itself; name the policy object involving it. A bill counts as a topic by its subject or name, never as a bare bill number such as "H.R. 1". A bare number or amount is never a topic. Entities mentioned only as examples, comparisons, or asides are not topics.

List at most {max_topics} topics. Include a topic only if the hearing gave it substantial attention — most summaries support 2 to 4 topics, and padding to {max_topics} is worse than stopping early. Every major subject of the hearing should appear.

--- SUMMARY ---
{summary}
--- END OF SUMMARY ---

Return only a JSON list of topic strings, most central first. Your very first output character must be [."""


def render_track1(summary):
    """The topic-listing prompt for one hearing summary."""
    return TRACK1_USER.format(max_topics=TRACK1_MAX_TOPICS, summary=summary)


# Directional words that must not appear inside a topic phrase; direction belongs in the stance field.
# The lint flags a phrase for review. It does not rewrite it.
DIRECTIONAL_LINT = re.compile(
    r"\b(?:ban(?:ning)?|block(?:ing)?|stop(?:ping)?|halt(?:ing)?|"
    r"oppos(?:e|ing|ition)|support(?:ing)?|defend(?:ing)?|protect(?:ing|ion)?|"
    r"against|anti|pro|promot(?:e|ing|ion)|expand(?:ing)?|restrict(?:ing|ion)?|"
    r"repeal(?:ing)?|eliminat(?:e|ing|ion))\b",
    re.IGNORECASE,
)


def lint_topic(phrase):
    """The directional word a topic phrase contains, or None if clean."""
    match = DIRECTIONAL_LINT.search(phrase or "")
    return match.group(0) if match else None


## Step 4. Define the model call and the log

One prompt, one completion, fresh context, near-greedy sampling. Every call is logged before its output is parsed, so a call that happened is on disk even if the parse fails.


In [ ]:
MAX_NEW_TOKENS = 256  # ample for five short phrases; hitting the cap is recorded as an error, not hidden

GENERATION = {"do_sample": True, "temperature": 0.1}  # near-greedy sampling, the same setting as T1


def render(tokenizer, system, user):
    """Apply the model's chat template, with the thinking mode off where the template offers it."""
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def generate(model, tokenizer, system, user, max_new_tokens):
    """One prompt, one completion, fresh context. Also returns whether the completion ran into the token cap."""
    prompt = render(tokenizer, system, user)
    ids = tokenizer(prompt, return_tensors="pt").to(model.device)
    n_in = ids["input_ids"].shape[-1]
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id, **GENERATION)
    generated = out[0][n_in:]
    hit_cap = len(generated) >= max_new_tokens
    return tokenizer.decode(generated, skip_special_tokens=True).strip(), hit_cap


class CallLog:
    """One JSON line per model call, written as it happens: stage, level, position, a hash of the exact input, token counts, time, and the full output. The same format as T1."""

    def __init__(self, path, document_id, model_name, prompt_version):
        self.path = Path(path) if path else None
        self.document_id = document_id
        self.model_name = model_name
        self.prompt_version = prompt_version
        self.records = []
        if self.path:
            self.path.parent.mkdir(parents=True, exist_ok=True)

    def write(self, stage, level, index, n_inputs, prompt_text, output, tokenizer, seconds):
        record = {
            "document_id": self.document_id, "model": self.model_name, "prompt_version": self.prompt_version,
            "stage": stage, "level": level, "index": index, "n_inputs": n_inputs,
            "input_sha256": hashlib.sha256(prompt_text.encode("utf-8")).hexdigest(),
            "input_chars": len(prompt_text),
            "input_tokens": len(tokenizer.encode(prompt_text, add_special_tokens=False)),
            "output_tokens": len(tokenizer.encode(output, add_special_tokens=False)),
            "output_chars": len(output), "seconds": round(seconds, 2),
            "at": datetime.now(timezone.utc).isoformat(timespec="seconds"), "output": output,
        }
        self.records.append(record)
        if self.path:
            with self.path.open("a") as log_file:
                log_file.write(json.dumps(record) + "\n")
                log_file.flush()


## Step 5. Define the parser

Every property that code can check is checked and counted. The output must be a JSON list of strings. Duplicates and topics beyond five are dropped, and the lint flag is attached.


In [ ]:
def parse_topics(output):
    """Validate one Track 1 output into ranked topics and a list of named errors.

    Case-insensitive duplicates are dropped and counted, the first occurrence winning; topics beyond
    TRACK1_MAX_TOPICS are dropped and counted; the directional lint keeps the topic with a flag.
    """
    errors = []
    text = output.strip()
    if text.startswith("```"):
        errors.append("markdown_fence")
        text = re.sub(r"^```[a-zA-Z]*\s*|\s*```$", "", text).strip()
    try:
        raw = json.loads(text)
    except json.JSONDecodeError:
        return [], errors + ["json_parse"]
    if not isinstance(raw, list):
        return [], errors + ["not_a_list"]
    topics, seen = [], set()
    for item in raw:
        if not (isinstance(item, str) and item.strip()):
            errors.append("topic_not_string")
            continue
        phrase = item.strip()
        if phrase.lower() in seen:
            errors.append("duplicate_topic")
            continue
        if len(topics) >= TRACK1_MAX_TOPICS:
            errors.append("too_many_topics")
            continue
        seen.add(phrase.lower())
        topics.append({"rank": len(topics) + 1, "topic": phrase, "topic_lint": lint_topic(phrase)})
    return topics, errors


## Step 6. List the hearings to run

Every hearing with a summary. A hearing is finished when its row is in the output table, so an interrupted run resumes where it stopped.


In [1]:
MODEL = os.environ.get("MODEL", "Qwen/Qwen3.5-4B")
TRACK1_FOLDER.mkdir(parents=True, exist_ok=True)
print(f"model   : {MODEL}")
print(f"prompt  : {TRACK1_PROMPT_VERSION}")
print(f"cap     : {MAX_NEW_TOKENS} new tokens, at most {TRACK1_MAX_TOPICS} topics")
print(f"sampling: {GENERATION}")
print(f"gpu     : {torch.cuda.get_device_name(0)}")


model   : Qwen/Qwen3.5-4B
prompt  : summary-topics-0.2-dev
cap     : 256 new tokens, at most 5 topics
sampling: {'do_sample': True, 'temperature': 0.1}
gpu     : NVIDIA A40


In [2]:
summaries = pd.read_parquet(INPUT_FOLDER / "hearing_summaries.parquet", columns=["packageId", "year", "chamber", "title", "summary", "status"])
assert (summaries["status"] == "ok").all(), "T1 left summaries that are not ok"

rows = []
if OUT_FILE.exists():
    rows = pd.read_parquet(OUT_FILE).to_dict("records")
done = {r["packageId"] for r in rows}
todo = summaries.loc[~summaries["packageId"].isin(done)].reset_index(drop=True)

print(f"hearings  : {len(summaries):,}")
print(f"  done    : {len(done):,}")
print(f"  to run  : {len(todo):,}")


hearings  : 2,352
  done    : 0
  to run  : 2,352


## Step 7. Load the model


In [3]:
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype="auto", device_map="cuda")
model.eval()
print(f"loaded in {time.time() - t0:.0f}s, {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B parameters")


loaded in 4s — 4.21B params


## Step 8. Extract the topics

Each hearing seeds the sampler from a hash of its id. So the result of a hearing does not depend on the order of the run. The output table is rewritten every 100 hearings.


In [4]:
started = time.time()
for n, row in enumerate(todo.itertuples(), 1):
    torch.manual_seed(int.from_bytes(hashlib.sha256(row.packageId.encode()).digest()[:4], "big"))
    log = CallLog(LOG_FILE, row.packageId, MODEL, prompt_version=TRACK1_PROMPT_VERSION)

    user = render_track1(row.summary)
    t0 = time.time()
    output, hit_cap = generate(model, tokenizer, SYSTEM, user, max_new_tokens=MAX_NEW_TOKENS)
    log.write("track1", 0, 0, 1, user, output, tokenizer, time.time() - t0)
    topics, errors = parse_topics(output)
    if hit_cap:
        errors.append("hit_token_cap")

    rows.append({
        "packageId": row.packageId, "year": int(row.year), "chamber": row.chamber,
        "prompt_version": TRACK1_PROMPT_VERSION, "model": MODEL,
        "n_topics": len(topics), "topics_json": json.dumps(topics), "errors": ";".join(errors),
    })
    if n % 100 == 0 or n == len(todo):
        pd.DataFrame(rows).to_parquet(OUT_FILE, compression="zstd", index=False)
        rate = n / (time.time() - started)
        print(f"  {n:>5,}/{len(todo):,}  {rate:.2f} hearings/s  eta {(len(todo) - n) / rate / 60:.0f} min")

pd.DataFrame(rows).to_parquet(OUT_FILE, compression="zstd", index=False)
print(f"\n{len(rows):,} hearings, {(time.time() - started) / 60:.0f} min this session")


    100/2,352  0.42 hearings/s  eta 88 min


    200/2,352  0.41 hearings/s  eta 87 min


    300/2,352  0.41 hearings/s  eta 83 min


    400/2,352  0.41 hearings/s  eta 80 min


    500/2,352  0.41 hearings/s  eta 76 min


    600/2,352  0.41 hearings/s  eta 72 min


... (14 outputs left out)

  2,100/2,352  0.41 hearings/s  eta 10 min


  2,200/2,352  0.41 hearings/s  eta 6 min


  2,300/2,352  0.41 hearings/s  eta 2 min


  2,352/2,352  0.41 hearings/s  eta 0 min

2,352 hearings, 96 min this session


## Step 9. Check the run

- Every hearing has one row and every row has a logged call.
- The error rate stays under 5 percent.
- The count of topics per hearing is printed, and so is the lint count, which marks phrases for review.


In [5]:
back = pd.read_parquet(OUT_FILE)
assert not back.duplicated(["packageId"]).any(), "hearing done twice"
assert len(back) == len(summaries), "coverage incomplete"

logged_ids = set()
with open(LOG_FILE) as f:
    n_log_lines = 0
    for line in f:
        n_log_lines += 1
        logged_ids.add(json.loads(line)["document_id"])
unlogged = set(back["packageId"]) - logged_ids
assert not unlogged, f"rows with no logged call: {sorted(unlogged)[:5]}"

err_rate = (back["errors"] != "").mean()
all_topics = [t for js in back["topics_json"] for t in json.loads(js)]
lint = sum(1 for t in all_topics if t.get("topic_lint"))

print(f"1) hearings       : {len(back):,} of {len(summaries):,}, no duplicates")
print(f"2) call log       : {n_log_lines:,} lines, every row backed by a logged call")
print(f"3) error rate     : {err_rate:.1%} [{'OK' if err_rate < 0.05 else 'OUT OF BAND'}]")
print(f"4) topics/hearing : {back['n_topics'].value_counts().sort_index().to_dict()}")
print(f"5) total phrases  : {len(all_topics):,} for the phrase pool")
print(f"6) topic lint     : {lint:,} flagged ({lint / max(len(all_topics), 1):.1%}), a review queue, not a removal")
print(f"7) prompt version : {back['prompt_version'].unique().tolist()}")


1) hearings       : 2,352 of 2,352, no duplicates
2) call log       : 2,352 lines, every row backed by a logged call
3) error rate     : 0.6% [OK]
4) topics/hearing : {2: 2, 3: 9, 4: 144, 5: 2197}
5) total phrases  : 11,592 for the pooled phrase set
6) topic lint     : 219 flagged (1.9%) — review queue, not removal
7) prompt version : ['summary-topics-0.2-dev']
